# Concentration Inequalities

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 09.03 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/09_Advanced_Probability/04_concentration_inequalities.ipynb)

---

## 🎯 Learning Objective

Learn Hoeffding, Bernstein, and McDiarmid concentration bounds, and see how they lead to PAC-Bayes generalization guarantees.

---

## 📐 Theory

The Law of Large Numbers (recall `03.01`-`03.03`) says a sample average converges to the true
mean as $n\to\infty$ — but says nothing about *how fast*, or how likely a large deviation is at
any *finite* $n$. **Concentration inequalities** answer exactly that: quantitative bounds on how
tightly a random quantity clusters around its expectation, for a *specific* sample size.

### Hoeffding's inequality

For independent random variables $X_1,\ldots,X_n$ each bounded in $[a,b]$, with sample mean
$\bar{X} = \frac1n\sum_i X_i$ and true mean $\mu = \mathbb{E}[\bar X]$, **Hoeffding's
inequality** bounds the probability of a large deviation:

$$P(|\bar X - \mu| \ge t) \le 2\exp\!\left(-\frac{2nt^2}{(b-a)^2}\right)$$

Two structural facts matter for how you'll use this: the bound decays *exponentially* in $n$
(doubling your sample size doesn't just modestly tighten the bound, it squares it), and it only
needs boundedness, not any assumption about the underlying distribution's shape — a genuinely
distribution-free guarantee.

### Bernstein's inequality: using variance, not just range

Hoeffding only uses the range $b-a$; **Bernstein's inequality** additionally incorporates the
variance $\sigma^2$ of each $X_i$, giving a tighter bound when the true variance is small
relative to the range:

$$P(|\bar X - \mu| \ge t) \le 2\exp\!\left(-\frac{nt^2/2}{\sigma^2 + \frac{(b-a)t}{3}}\right)$$

For small $t$, the denominator is dominated by $\sigma^2$ rather than the range, so Bernstein
can be substantially sharper than Hoeffding whenever the data is low-variance even if its
theoretical range is wide.

### McDiarmid's inequality: beyond simple averages

Hoeffding and Bernstein bound deviations of an *average*. **McDiarmid's inequality**
generalizes to any function $f(X_1,\ldots,X_n)$ satisfying the **bounded differences**
property — changing any single $X_i$ (holding the rest fixed) can change $f$ by at most $c_i$:

$$P(|f(X_1,\ldots,X_n) - \mathbb{E}[f]| \ge t) \le 2\exp\!\left(-\frac{2t^2}{\sum_i c_i^2}\right)$$

This covers far more than sample means — it applies to *any* sufically-stable statistic
computed from independent data, including many quantities that show up when analyzing a
trained model's behavior across a random training set.

### Chernoff bounds

**Chernoff bounds** are a general technique (rather than a single named inequality):
bound $P(X\ge t)$ by exponentiating and applying Markov's inequality to $e^{\lambda X}$ for an
optimally-chosen $\lambda$. Hoeffding's inequality is itself derived via a Chernoff-style
argument specialized to bounded random variables — the "family," not a separate result.

### From concentration to generalization: PAC-style bounds

In supervised learning, you observe *training* error $\hat{L}(\theta)$ (an average over $n$
training points) but care about *true* error $L(\theta)$ (an average over the entire, usually
infinite, data distribution). Hoeffding's inequality — applied with $X_i$ = the (bounded)
per-example loss — converts this directly into a **generalization bound**: with probability at
least $1-\delta$,

$$L(\theta) \le \hat{L}(\theta) + \sqrt{\frac{\ln(2/\delta)}{2n}}$$

(solving Hoeffding's tail bound for $t$ given a target confidence $\delta$). This is the
essential shape of every PAC ("Probably Approximately Correct") learning-theory result: with
high probability ($1-\delta$), the true error is close to the observed training error, by an
amount that shrinks as $\sqrt{1/n}$. **PAC-Bayes** bounds extend this to hold *simultaneously*
across an entire (possibly infinite) family of hypotheses weighted by a prior, at the cost of an
extra term measuring how far the learned "posterior" over hypotheses has moved from that prior —
conceptually the same generalization-vs-training-size tradeoff, now accounting for model
complexity rather than treating a single fixed $\theta$ in isolation.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Overlaying the empirical exceedance probability (from repeated simulation) against Hoeffding's
theoretical bound, across a range of deviation thresholds, shows the bound is a genuine
*upper* envelope — never crossed, sometimes loose, but never violated.

In [ ]:
def hoeffding_bound(n, t, a=0, b=1):
    return np.minimum(1.0, 2 * np.exp(-2 * n * t**2 / (b - a)**2))

rng = np.random.default_rng(0)
n, true_mean, n_trials = 200, 0.3, 20000
samples_all = rng.uniform(size=(n_trials, n)) < true_mean   # Bernoulli(0.3), bounded in [0,1]
sample_means = samples_all.mean(axis=1)
deviations = np.abs(sample_means - true_mean)

t_grid = np.linspace(0.01, 0.3, 60)
empirical_probs = np.array([np.mean(deviations > t) for t in t_grid])
bound_probs = hoeffding_bound(n, t_grid)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.semilogy(t_grid, empirical_probs, 'o-', color="#4C72B0", markersize=3, label="empirical P(|mean-true|>t)")
ax.semilogy(t_grid, bound_probs, color="#C44E52", lw=2, label="Hoeffding bound")
ax.set_xlabel("deviation threshold t"); ax.set_ylabel("probability (log scale)")
ax.set_title(f"Hoeffding's bound (n={n}) never crosses below\nthe true empirical exceedance probability")
ax.legend()
plt.show()

violations = np.sum(empirical_probs > bound_probs)
print(f"Number of thresholds where empirical probability EXCEEDS the bound: {violations} (out of {len(t_grid)})")
print(f"Confirms Hoeffding's inequality is a genuine, never-violated upper bound -- not just a rough heuristic.")

## 🐍 Implementation from Scratch

We implement Hoeffding's and Bernstein's bounds directly from their formulas, stress-test
Hoeffding's bound across many random trials to confirm it's never violated, and compare the two
bounds' tightness on a low-variance bounded random variable.

In [ ]:
def hoeffding_bound(n, t, a=0, b=1):
    return np.minimum(1.0, 2 * np.exp(-2 * n * t**2 / (b - a)**2))

def bernstein_bound(n, t, sigma2, a=0, b=1):
    return np.minimum(1.0, 2 * np.exp(-(n * t**2 / 2) / (sigma2 + (b - a) * t / 3)))

# --- Stress-test: does Hoeffding's bound ever get violated across MANY thresholds/trials? ---
rng = np.random.default_rng(0)
n, true_mean, n_trials = 200, 0.3, 20000
samples = rng.uniform(size=(n_trials, n)) < true_mean   # Bernoulli(0.3), bounded in [0, 1]
deviations = np.abs(samples.mean(axis=1) - true_mean)

t_grid = np.linspace(0.01, 0.3, 100)
empirical = np.array([np.mean(deviations > t) for t in t_grid])
bound = hoeffding_bound(n, t_grid)
violations = np.sum(empirical > bound)
print(f"Hoeffding bound violated at {violations} / {len(t_grid)} thresholds tested")
print(f"(0 violations confirms it's a genuine upper bound, not a rough approximation)")

# --- Bernstein vs. Hoeffding: does using the variance actually help on a LOW-variance case? ---
low_var_samples = rng.beta(50, 50, size=(20000, n))  # concentrates near 0.5, but range is still [0,1]
sigma2 = low_var_samples.var()
true_mean_beta = 0.5
deviations_beta = np.abs(low_var_samples.mean(axis=1) - true_mean_beta)

print(f"\nEmpirical per-sample variance: {sigma2:.5f}  (range [0,1], so Hoeffding can't see this)")
print(f"{'t':>6} {'empirical P':>12} {'Hoeffding bound':>16} {'Bernstein bound':>16}")
for t in [0.02, 0.05, 0.08]:
    emp = np.mean(deviations_beta > t)
    hb, bb = hoeffding_bound(n, t), bernstein_bound(n, t, sigma2)
    print(f"{t:6.2f} {emp:12.5f} {hb:16.5f} {bb:16.5f}")
print("\nBernstein's bound is dramatically tighter here because it uses the ACTUAL (small)")
print("variance, while Hoeffding only sees the (much larger) theoretical range [0,1].")

## 🤖 Why This Matters for AI

Every claim of the form "this model achieves X% test accuracy with 95% confidence" is a
concentration-inequality statement, whether or not the paper spells out which inequality. The
PAC-style bound derived above turns Hoeffding's inequality directly into a **generalization
bound**: given a model's *observed* training loss and the training set size $n$, you get a
principled (if often loose) upper bound on the *true* population loss, with a specific
confidence level — exactly the theoretical backbone behind sample-complexity arguments ("how
much data do we need to trust this model's reported accuracy?"). Below, we compute a concrete
generalization bound for a toy classifier's observed training accuracy, and confirm it via
direct simulation from a known ground-truth distribution: does the true accuracy really fall
within the bound's guarantee, across many random training sets?

The full generalization-theory toolkit that builds on these inequalities — **PAC-Bayes bounds**, **VC dimension**, and **Rademacher complexity** — is developed in Module 13 (`13.04`–`13.06`), where concentration inequalities serve as the foundational machinery for every sample-complexity and generalization result.

---


In [ ]:
def pac_generalization_gap(n, delta):
    # Hoeffding's tail bound, solved for t given a target confidence 1-delta
    return np.sqrt(np.log(2 / delta) / (2 * n))

# A model with some FIXED, unknown true accuracy against the real data distribution --
# we simulate it as 0.82 so we can check the bound's promise against ground truth
true_accuracy = 0.82
n_train, delta = 500, 0.05   # want the bound to hold with probability >= 95%

gap = pac_generalization_gap(n_train, delta)
print(f"With n={n_train} training examples and delta={delta} (95% confidence):")
print(f"generalization gap = {gap:.4f}")
print(f"If observed training accuracy is, say, 0.85, the bound guarantees (w.p. >= 95%)")
print(f"true accuracy is in [{0.85 - gap:.4f}, {0.85 + gap:.4f}]")

# Does the bound actually deliver on its promise? Simulate MANY random training sets from
# the SAME true distribution and check how often the true accuracy falls in the claimed interval
n_trials = 3000
covered = 0
for _ in range(n_trials):
    correct = rng.uniform(size=n_train) < true_accuracy
    observed_acc = correct.mean()
    if observed_acc - gap <= true_accuracy <= observed_acc + gap:
        covered += 1

coverage_rate = covered / n_trials
print(f"\nEmpirical coverage rate over {n_trials} simulated training runs: {coverage_rate:.4f}")
print(f"Bound promised at least: {1 - delta:.2f}")
print(f"Promise kept: {coverage_rate >= 1 - delta}  (Hoeffding-based bounds are typically")
print(f"CONSERVATIVE -- true coverage often exceeds the promised minimum, as it does here.)")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $n=1000$ samples of a $[0,1]$-bounded random variable, compute Hoeffding's bound at
   $t=0.05$ by hand. Check your result is a valid probability (between $0$ and $1$) before
   verifying with `hoeffding_bound`.

<details>
<summary>💡 Solution</summary>

$2\exp\left(-\frac{2\cdot1000\cdot0.05^2}{(1-0)^2}\right) = 2\exp(-5) \approx 0.01348$ — a
valid probability (between $0$ and $1$, as any bound on a probability must be). Calling
`hoeffding_bound(1000, 0.05)` returns the identical value.

</details>

### 💭 UNDERSTAND
2. The Visual Intuition cell reports "0 violations" of Hoeffding's bound across 60 thresholds
   and 20,000 trials, and frames this as confirming the bound is "never crossed." Someone argues
   this is circular: since the bound is a mathematical theorem, of course a simulation will
   never contradict it, so why bother running the check at all? Push back on this: what could
   the simulation have actually revealed if something were wrong, and why is a mathematically
   *proven* bound still worth numerically checking in code?

<details>
<summary>💡 Solution</summary>

The simulation isn't testing whether the mathematical theorem is true (it is, and no amount of
simulation could add or subtract confidence in a proof) — it's testing whether the *code*
correctly implements the theorem's formula and applies it under the theorem's actual
assumptions (independence, boundedness in $[a,b]$). A bug in `hoeffding_bound` (e.g. a wrong
exponent, or an off-by-one in $n$), or accidentally simulating dependent rather than independent
samples, or samples outside $[0,1]$, would all plausibly produce violations in exactly this kind
of check even though the *theorem* remains true — the check is validating the implementation and
its assumptions, not re-deriving the theorem. This is the same "verify by an independent method"
discipline used throughout the repo's from-scratch implementations, applied to a proven
inequality rather than a computed value.

</details>

### ✏️ DERIVE
3. Bernstein's bound reduces to something Hoeffding-like in a limiting case. Using the
   Bernstein formula $P(|\bar X-\mu|\ge t)\le2\exp\left(-\frac{nt^2/2}{\sigma^2+(b-a)t/3}\right)$,
   derive what happens to the exponent as $\sigma^2 \to \left(\frac{b-a}{2}\right)^2$ (the
   maximum possible variance for a $[a,b]$-bounded variable, achieved by a two-point
   distribution at the endpoints) and small $t$, and compare the resulting bound's *rate* of
   decay in $n$ to Hoeffding's own $n$-dependence.

<details>
<summary>💡 Solution outline</summary>

For small $t$, the $(b-a)t/3$ term in the denominator becomes negligible compared to $\sigma^2$,
so the exponent approaches $-\frac{nt^2}{2\sigma^2}$. Both this limiting form and Hoeffding's
exponent $-\frac{2nt^2}{(b-a)^2}$ share the same *structure*: linear in $n$, quadratic in $t$ —
so both bounds decay exponentially in $n$ at the same asymptotic *rate* (in $n$), differing only
in the constant multiplying $n$. This constant is where Bernstein's advantage lives: when
$\sigma^2$ is much smaller than $\left(\frac{b-a}{2}\right)^2$ (low-variance data, as in the
Implementation cell's `beta(50,50)` example), Bernstein's exponent decays much faster per unit
of $n$ than Hoeffding's fixed, variance-blind constant — the two bounds are the same *family* of
result, but Bernstein adapts to the data's actual variance while Hoeffding can only ever use the
worst-case variance implied by the range.

</details>

### 🐍 IMPLEMENT
4. In the Bernstein-vs-Hoeffding comparison, replace `rng.beta(50, 50, ...)` with
   `rng.beta(2, 2, ...)` (higher variance, still bounded on $[0,1]$) and recompute both bounds
   at $t=0.05$. Does Bernstein's advantage over Hoeffding shrink, grow, or stay about the same?

<details>
<summary>✅ How to know you're right</summary>

At `beta(50,50)` (empirical variance $\approx0.00247$), Hoeffding's bound is $\approx0.736$
while Bernstein's is astronomically smaller ($\approx3\times10^{-6}$, a ratio of
$\approx173{,}000\times$). At `beta(2,2)` (empirical variance $\approx0.050$, about $20\times$
larger), Hoeffding's bound stays exactly the same ($\approx0.736$, since Hoeffding never looks
at the variance at all), while Bernstein's bound grows substantially looser
($\approx0.047$) — Bernstein's advantage over Hoeffding shrinks from a factor of
$\approx173{,}000$ down to a factor of $\approx15.6$. This matches Exercise 3's derivation
directly: Bernstein's edge over Hoeffding comes specifically from a small $\sigma^2$, so
increasing the variance erodes that edge, even though Bernstein's bound remains valid (never
worse than trivial) throughout.

</details>

### 🤖 APPLY
5. McDiarmid's inequality applies to *any* function with the bounded-differences property, not
   just sample means. Verify this directly on $f(X_1,\ldots,X_n) = \max(X_1,\ldots,X_n)$ for
   $n=50$ iid $\text{Uniform}(0,1)$ variables: changing any single $X_i$ can change the max by
   at most $c_i=(b-a)=1$ (the full range), giving $\sum_i c_i^2 = n$. Confirm McDiarmid's bound
   is never violated for this genuinely non-average statistic, the way Hoeffding's bound was
   confirmed for sample means in the Implementation section.

<details>
<summary>✅ What you should observe</summary>

The empirical mean of the max of 50 uniforms should be close to the known formula
$n/(n+1)\approx0.9804$ (a useful independent sanity check before trusting the concentration
result itself). McDiarmid's bound, using $\sum_i c_i^2=50$, should show zero violations across a
grid of thresholds — but the bound will also be visibly *much looser* here than Hoeffding's was
for a sample mean (e.g. at $t=0.1$, the empirical exceedance probability might be under $1\%$
while McDiarmid's bound sits near $1.0$, essentially uninformative at that threshold). This is
the real tradeoff McDiarmid trades for its generality: applying to almost any bounded-difference
statistic, at the cost of using each $c_i$'s *worst-case* sensitivity (here, the full range) rather
than anything tailored to the specific statistic (a sample mean's sensitivity to one point is
only $\frac{b-a}{n}$, far smaller than the max's full $b-a$) — so the same machinery that made
Hoeffding tight for averages becomes much less tight for less-averaging-friendly statistics
like the max.

</details>

### 🚀 CHALLENGE
6. Increase `n_train` in the AI section from 500 to 5000 (a $10\times$ increase) and re-run the
   coverage simulation. Report the new generalization gap and empirical coverage rate, and check
   whether the gap shrank by the factor the Theory section's $\sqrt{1/n}$ scaling predicts.

<details>
<summary>🔍 What a strong answer covers</summary>

At `n_train=500`, the gap is $\approx0.0607$; at `n_train=5000`, it drops to $\approx0.0192$ — a
ratio of $\approx3.162$, matching $\sqrt{10}\approx3.162$ (the factor predicted by $\sqrt{1/n}$
scaling for a $10\times$ increase in $n$) essentially exactly, since the gap formula
$\sqrt{\ln(2/\delta)/(2n)}$ is an *exact* $1/\sqrt{n}$ function of $n$ with no approximation
involved — this isn't a rough asymptotic tendency but a precise algebraic consequence of the
formula. The empirical coverage rate at `n_train=5000` comes out around $0.9997$, still safely
above the promised $0.95$ (and, consistent with the notebook's own note, on the conservative
side — Hoeffding-based PAC bounds routinely over-deliver on their stated confidence level rather
than sitting exactly at it). A strong answer connects this to the diminishing-returns character
of the $\sqrt{1/n}$ law: getting the NEXT $10\times$ reduction in gap (from $\approx0.019$ down
to $\approx0.006$) would require another full $10\times$ increase in training data, to
$n=50{,}000$ — tightening a generalization guarantee gets increasingly data-hungry, not
proportionally cheaper, as the target gap shrinks.

</details>

---

## 📚 Further Reading
- Boucheron, Lugosi & Massart, *Concentration Inequalities: A Nonasymptotic Theory of Independence*
- McAllester, ["PAC-Bayesian Model Averaging"](https://www.cs.cmu.edu/~avrim/ML05/pdf/mcallester.pdf)
- Shalev-Shwartz & Ben-David, *Understanding Machine Learning*, Ch. 4 (PAC Learning)

---

*Next notebook: [Generative Model Theory](05_generative_model_theory.ipynb)*